# Paso 3: Exploracion y limpieza

Proyecto: evaluacion de salud/riesgo de repositorios de GitHub (GH Archive).

**Actualizacion (Paso 7):** esto ya lo habia corrido una vez con las 12
horas originales. Ahora que descargamos ~124 horas (12 originales + 57
dispersas nuevas + un bloque de 4 dias seguidos), vuelvo a correr todo el
notebook desde cero sobre el dataset grande. Como el codigo ya lee con un
comodin (`*.json.gz`), no tengo que cambiar la logica, solo re-ejecutar y
ver si los numeros/patrones cambian con mas datos.

En esta fase:
1. Cargamos todas las horas crudas de `data/raw/` (capa **Bronze**).
2. Exploramos los datos: cuantos eventos hay por tipo y por mes (con la
   version chica del dataset habiamos visto un patron raro: de marzo 2026
   en adelante bajaban mucho los Issues/PR/Fork/Watch mientras subia el
   Push -- veamos si se sostiene con mas horas por mes).
3. Limpiamos (capa **Silver**): nos quedamos solo con los 5 tipos de evento
   de interes, quitamos bots, quitamos duplicados y filas con campos nulos
   criticos (repo, actor, fecha).
4. Guardamos el resultado limpio en `data/processed/` para usarlo en la
   siguiente fase (features + clustering).

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, substring

# Sesion local, sin cluster real (todo corre en esta misma maquina).
# Con las 12 horas originales (~477MB) 4g de driver alcanzaba. Ahora con
# ~124 horas (~3.9GB comprimidos) subo a 6g -- es lo maximo razonable que
# puedo darle sin ahogar el resto de la maquina (tengo ~6GB libres). Si
# esto explota con OutOfMemory voy a tener que procesar por partes.
spark = (SparkSession.builder
         .master("local[*]")
         .appName("limpieza-gharchive")
         .config("spark.driver.memory", "6g")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/11 22:57:40 WARN Utils: Your hostname, katana, resolves to a loopback address: 127.0.1.1; using 192.168.18.59 instead (on interface enp2s0)
26/07/11 22:57:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/07/11 22:57:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Capa Bronze: cargar los archivos crudos

Leemos todos los `.json.gz` de `data/raw/` en un solo DataFrame de Spark
(ya no son solo 12, ahora son ~124). Cada linea del archivo es un evento
en JSON, Spark los infiere solo.

In [2]:
# El comodin *.json.gz toma los 12 archivos de una sola vez
df_bronze = spark.read.json("../data/raw/*.json.gz")

print("total de eventos (bronze, sin filtrar):", df_bronze.count())
df_bronze.printSchema()

total de eventos (bronze, sin filtrar): 17909186
root
 |-- actor: struct (nullable = true)
 |    |-- avatar_url: string (nullable = true)
 |    |-- display_login: string (nullable = true)
 |    |-- gravatar_id: string (nullable = true)
 |    |-- id: long (nullable = true)
 |    |-- login: string (nullable = true)
 |    |-- url: string (nullable = true)
 |-- created_at: string (nullable = true)
 |-- id: string (nullable = true)
 |-- org: struct (nullable = true)
 |    |-- avatar_url: string (nullable = true)
 |    |-- gravatar_id: string (nullable = true)
 |    |-- id: long (nullable = true)
 |    |-- login: string (nullable = true)
 |    |-- url: string (nullable = true)
 |-- payload: struct (nullable = true)
 |    |-- action: string (nullable = true)
 |    |-- assignee: struct (nullable = true)
 |    |    |-- avatar_url: string (nullable = true)
 |    |    |-- events_url: string (nullable = true)
 |    |    |-- followers_url: string (nullable = true)
 |    |    |-- following_url: stri

## Exploracion: revisar el patron raro por mes

En el Paso 2 notamos que de marzo 2026 en adelante los conteos de
Issues/PR/Fork/Watch bajaban mucho mientras Push subia. Aca lo confirmamos
agrupando por mes y tipo de evento, sobre los 5 tipos de interes (todavia
sin quitar bots, para ver el dato crudo).

In [3]:
EVENTOS_DE_INTERES = ["PushEvent", "IssuesEvent", "PullRequestEvent", "WatchEvent", "ForkEvent"]

df_interes = df_bronze.filter(col("type").isin(EVENTOS_DE_INTERES))

# "created_at" viene como texto tipo 2026-06-16T19:00:00Z -> los primeros
# 7 caracteres son el mes (AAAA-MM)
df_interes = df_interes.withColumn("mes", substring(col("created_at"), 1, 7))

(df_interes
    .groupBy("mes", "type")
    .agg(count("*").alias("cantidad"))
    .orderBy("mes", "type")
    .show(60))

+-------+----------------+--------+
|    mes|            type|cantidad|
+-------+----------------+--------+
|2025-08|       ForkEvent|    7535|
|2025-08|     IssuesEvent|   20288|
|2025-08|PullRequestEvent|   70106|
|2025-08|       PushEvent|  584123|
|2025-08|      WatchEvent|   28581|
|2025-09|       ForkEvent|    7085|
|2025-09|     IssuesEvent|   18254|
|2025-09|PullRequestEvent|   77107|
|2025-09|       PushEvent|  609442|
|2025-09|      WatchEvent|   27426|
|2025-10|       ForkEvent|    6262|
|2025-10|     IssuesEvent|   21356|
|2025-10|PullRequestEvent|   69694|
|2025-10|       PushEvent|  606142|
|2025-10|      WatchEvent|   23842|
|2025-11|       ForkEvent|    5615|
|2025-11|     IssuesEvent|   23413|
|2025-11|PullRequestEvent|   58559|
|2025-11|       PushEvent|  620061|
|2025-11|      WatchEvent|   20579|
|2025-12|       ForkEvent|    3909|
|2025-12|     IssuesEvent|   19346|
|2025-12|PullRequestEvent|   70710|
|2025-12|       PushEvent|  621062|
|2025-12|      WatchEvent|  

### Hallazgo: junio 2026 esta "roto" en la tabla de arriba, y es culpa nuestra

Mirando la tabla, `2026-06` tiene un PushEvent de ~6.9 millones, muchisimo
mas que cualquier otro mes (que rondan 580k-750k). Al principio pense que
era un patron real, pero no: es porque junio es el mes donde metimos el
bloque de 4 dias seguidos (56 horas) ADEMAS de las horas dispersas, asi
que junio tiene ~62 horas muestreadas contra solo 6 de los demas meses (y
julio tiene menos todavia, porque cortamos ahi por la fecha de "hoy").

O sea: comparar TOTALES por mes ya no es justo, porque no todos los meses
tienen la misma cantidad de horas muestreadas. Para comparar de verdad hay
que normalizar por cuantas horas se muestrearon en cada mes (eventos por
hora), no por el total. Abajo lo recalculo asi.

In [4]:
from pyspark.sql.functions import countDistinct

# "hora_bucket" identifica la hora exacta de origen (ej. 2026-06-03T13),
# que es basicamente el nombre del archivo de GH Archive del que vino el
# evento. Contando cuantas "hora_bucket" distintas hay por mes, sabemos
# cuantas horas muestreamos ese mes.
df_interes = df_interes.withColumn("hora_bucket", substring(col("created_at"), 1, 13))

horas_por_mes = df_interes.groupBy("mes").agg(countDistinct("hora_bucket").alias("horas_muestreadas"))

conteo_por_mes_tipo = df_interes.groupBy("mes", "type").agg(count("*").alias("cantidad"))

normalizado = (conteo_por_mes_tipo
               .join(horas_por_mes, on="mes")
               .withColumn("eventos_por_hora", col("cantidad") / col("horas_muestreadas")))

normalizado.select("mes", "type", "horas_muestreadas", "cantidad", "eventos_por_hora").orderBy("mes", "type").show(60)

+-------+----------------+-----------------+--------+------------------+
|    mes|            type|horas_muestreadas|cantidad|  eventos_por_hora|
+-------+----------------+-----------------+--------+------------------+
|2025-08|       ForkEvent|                6|    7535|1255.8333333333333|
|2025-08|     IssuesEvent|                6|   20288|3381.3333333333335|
|2025-08|PullRequestEvent|                6|   70106|11684.333333333334|
|2025-08|       PushEvent|                6|  584123| 97353.83333333333|
|2025-08|      WatchEvent|                6|   28581|            4763.5|
|2025-09|       ForkEvent|                6|    7085|1180.8333333333333|
|2025-09|     IssuesEvent|                6|   18254|3042.3333333333335|
|2025-09|PullRequestEvent|                6|   77107|12851.166666666666|
|2025-09|       PushEvent|                6|  609442|101573.66666666667|
|2025-09|      WatchEvent|                6|   27426|            4571.0|
|2025-10|       ForkEvent|                6|    626

### El patron se sostiene normalizado -- esto ya no parece casualidad

Con eventos por hora (no totales), el patron sigue ahi e incluso se ve
mas fuerte:

- `PushEvent` por hora sube de ~97,354 (ago 2025) a ~145,487 (jul 2026).
- `ForkEvent` por hora cae de ~1,256 a ~20 (-98%).
- `WatchEvent` por hora cae de ~4,764 a ~64 (-99%).
- `PullRequestEvent` por hora cae de ~11,684 a ~283 (-98%).
- `IssuesEvent` por hora cae de ~3,381 a ~94 (-97%).

Osea que no es un problema de muestreo (ya lo descarte normalizando) ni
un bug mio (mismo criterio, mismos 5 tipos de evento, en 124 horas
distintas). Es un patron real y fuerte en los datos: cada vez hay
proporcionalmente MAS pushes y MENOS de todo lo demas (forks, watches,
issues, PRs).

No se bien por que pasa esto -- se me ocurren varias hipotesis (mas
actividad automatizada/bots de agentes de IA haciendo push sin abrir
PRs/issues, algun cambio en como GH Archive registra estos eventos, etc.)
pero no tengo forma de confirmar la causa con estos datos solos. Lo dejo
anotado como hallazgo principal para discutir en la sustentacion, sin
inventar una explicacion que no pueda sustentar.

## Capa Silver: limpieza

Pasos, en orden (mostramos cuantas filas quedan despues de cada uno):

1. Quedarnos solo con los 5 tipos de evento de interes (ya lo hicimos arriba).
2. Quitar filas con campos nulos criticos: repo, actor o fecha (asi el
   filtro de bots del siguiente paso no se confunde con actores nulos).
3. Quitar bots (login termina en `-bot`, `[bot]`, o contiene `dependabot`).
4. Quitar eventos duplicados (cada evento de GH Archive trae un `id` unico).

In [5]:
print("1. despues de filtrar tipos de interes:", df_interes.count())

# Parte 2: nulos criticos (repo, actor, fecha)
df_sin_nulos = df_interes.filter(
    col("repo.name").isNotNull()
    & col("actor.login").isNotNull()
    & col("created_at").isNotNull()
)
print("2. despues de quitar nulos criticos:", df_sin_nulos.count())

1. despues de filtrar tipos de interes: 14935939


2. despues de quitar nulos criticos: 14935846


In [6]:
from pyspark.sql.functions import lower

# Parte 3: quitar bots (mismo criterio que usamos en src/ingest.py en Paso 1)
login = lower(col("actor.login"))
es_bot = login.endswith("-bot") | login.endswith("[bot]") | login.contains("dependabot")

df_sin_bots = df_sin_nulos.filter(~es_bot)
print("3. despues de quitar bots:", df_sin_bots.count())

3. despues de quitar bots: 12048304


In [7]:
# Parte 4: quitar duplicados exactos por id de evento (cada evento de
# GH Archive deberia tener un id unico)
df_silver = df_sin_bots.dropDuplicates(["id"])
print("4. despues de quitar duplicados:", df_silver.count())

4. despues de quitar duplicados: 12047086


## Guardar la capa Silver

Guardamos en formato Parquet (formato estandar de Spark, mas rapido de leer
que JSON) para usarlo en la fase de features/clustering.

In [8]:
RUTA_SALIDA = "../data/processed/eventos_silver.parquet"

df_silver.write.mode("overwrite").parquet(RUTA_SALIDA)

# Verificacion rapida: releer y contar
verificacion = spark.read.parquet(RUTA_SALIDA)
print("filas guardadas en", RUTA_SALIDA, ":", verificacion.count())

filas guardadas en ../data/processed/eventos_silver.parquet : 12047086
